In [4]:
from collections import defaultdict

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import poisson

# Load trained models
home_model = sm.load("models/poisson_elo_home_goals.pickle")
away_model = sm.load("models/poisson_elo_away_goals.pickle")

# Load match history used to create current team form and Elo ratings
matches = pd.read_csv(
    "data/processed/epl_matches_clean.csv",
    parse_dates=["Date"]
).sort_values("Date")

feature_columns = [
    "home_form_points_5",
    "away_form_points_5",
    "home_avg_goals_for_5",
    "away_avg_goals_for_5",
    "home_avg_goals_against_5",
    "away_avg_goals_against_5",
    "elo_difference",
]

team_history = defaultdict(list)
ratings = defaultdict(lambda: 1500.0)

K_FACTOR = 20
HOME_ADVANTAGE = 60

# Rebuild each team's form and Elo rating through the latest known match
for _, match in matches.iterrows():
    home_team = match["HomeTeam"]
    away_team = match["AwayTeam"]

    home_goals = match["FTHG"]
    away_goals = match["FTAG"]

    if home_goals > away_goals:
        home_points, away_points = 3, 0
        actual_home = 1.0
    elif home_goals < away_goals:
        home_points, away_points = 0, 3
        actual_home = 0.0
    else:
        home_points, away_points = 1, 1
        actual_home = 0.5

    # Update Elo after the result
    expected_home = 1 / (
        1 + 10 ** ((ratings[away_team] - (ratings[home_team] + HOME_ADVANTAGE)) / 400)
    )

    ratings[home_team] += K_FACTOR * (actual_home - expected_home)
    ratings[away_team] += K_FACTOR * ((1 - actual_home) - (1 - expected_home))

    # Update recent-match history
    team_history[home_team].append({
        "goals_for": home_goals,
        "goals_against": away_goals,
        "points": home_points,
    })

    team_history[away_team].append({
        "goals_for": away_goals,
        "goals_against": home_goals,
        "points": away_points,
    })

def get_recent_stats(team, last_n=5):
    recent = team_history[team][-last_n:]

    if len(recent) < last_n:
        raise ValueError(f"Not enough match history for {team}.")

    return {
        "form_points_5": sum(game["points"] for game in recent),
        "avg_goals_for_5": np.mean([game["goals_for"] for game in recent]),
        "avg_goals_against_5": np.mean([game["goals_against"] for game in recent]),
    }

def predict_match(home_team, away_team):
    home = get_recent_stats(home_team)
    away = get_recent_stats(away_team)

    match_features = pd.DataFrame([{
        "home_form_points_5": home["form_points_5"],
        "away_form_points_5": away["form_points_5"],
        "home_avg_goals_for_5": home["avg_goals_for_5"],
        "away_avg_goals_for_5": away["avg_goals_for_5"],
        "home_avg_goals_against_5": home["avg_goals_against_5"],
        "away_avg_goals_against_5": away["avg_goals_against_5"],
        "elo_difference": ratings[home_team] - ratings[away_team],
    }])

    X = sm.add_constant(
        match_features[feature_columns],
        has_constant="add"
    )

    home_expected_goals = home_model.predict(X).iloc[0]
    away_expected_goals = away_model.predict(X).iloc[0]

    score_rows = []

    for home_goals in range(8):
        for away_goals in range(8):
            probability = (
                poisson.pmf(home_goals, home_expected_goals) *
                poisson.pmf(away_goals, away_expected_goals)
            )

            score_rows.append({
                "Score": f"{home_goals}-{away_goals}",
                "Probability": probability,
                "HomeGoals": home_goals,
                "AwayGoals": away_goals,
            })

    scores = pd.DataFrame(score_rows)

    home_win = scores[scores["HomeGoals"] > scores["AwayGoals"]]["Probability"].sum()
    draw = scores[scores["HomeGoals"] == scores["AwayGoals"]]["Probability"].sum()
    away_win = scores[scores["HomeGoals"] < scores["AwayGoals"]]["Probability"].sum()

    top_scores = scores.sort_values("Probability", ascending=False).head(5).copy()
    top_scores["Probability"] = (top_scores["Probability"] * 100).round(1)

    print(f"{home_team} vs {away_team}")
    print(f"Expected goals: {home_team} {home_expected_goals:.2f} — {away_team} {away_expected_goals:.2f}")
    print(f"Home win: {home_win:.1%} | Draw: {draw:.1%} | Away win: {away_win:.1%}")

    return top_scores[["Score", "Probability"]]

print(f"Prediction data is current through: {matches['Date'].max().date()}")
print("\nExample teams:")
print(sorted(matches["HomeTeam"].unique()))

Prediction data is current through: 2026-09-20

Example teams:
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Coventry', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']


In [3]:
predict_match("Arsenal", "Chelsea")

Arsenal vs Chelsea
Expected goals: Arsenal 1.86 — Chelsea 0.92
Home win: 59.3% | Draw: 22.3% | Away win: 18.3%


,Score,Probability
8,1-0,11.5
16,2-0,10.7
9,1-1,10.6
17,2-1,9.9
24,3-0,6.6
